In [0]:
%run ../../0_Config/0-Init

In [0]:
def run_notebook(notebook_name):
    dbutils.notebook.run(notebook_name, 0)

In [0]:
job_list = [
    "001-atendimento_ocorrencias",
    "002-cadastro_produtos_api_dump",
    "003-comercial_canais",
    "004-crm_clientes_export",
    "005-erp_pedidos_cabecalho",
    "006-erp_pedidos_itens",
    "007-legado_regioes_pipe",
    "008-logistica_entregas",
    "009-vendedores"
]

In [0]:
from threading import Thread, Lock
from queue import Queue
from datetime import datetime, timedelta
import time

q = Queue()
results_lock = Lock()
resultados = []

worker_count = 9
timeout = 600  # 10 minutes in seconds

for job in job_list:
    q.put(job)

execution_start = datetime.now()

_fuso_print = timedelta(hours=-3)
print(f"\n{'='*55}")
print(f"  🚀 Iniciando execução de {len(job_list)} jobs | {worker_count} workers")
print(f"  📅 Início: {(execution_start + _fuso_print).strftime('%d/%m/%Y %H:%M:%S')}")
print(f"{'='*55}\n")


def run_tasks(function, q):
    while not q.empty():
        value = q.get()
        print(f"▶ Executando: {value}")
        start_time = time.time()
        error_msg = None
        timeout_msg = None
        status = "sucesso"

        try:
            run_notebook(value)
        except Exception as e:
            error_msg = str(e)
            status = "erro"
            print(f"  ✗ Erro: {value}")
            try:
                enviar_erro_teams(var_teams, value, time.time() - start_time, error_msg)
            except NameError:
                pass
        finally:
            elapsed_time = time.time() - start_time

            if elapsed_time > timeout and status != "erro":
                timeout_msg = f"Timeout ({elapsed_time:.0f}s > {timeout}s)"
                status = "timeout"
                print(f"  ⚠ Timeout: {value} ({elapsed_time:.0f}s)")
                try:
                    enviar_erro_teams(var_teams, value, elapsed_time, timeout_msg)
                except NameError:
                    pass

            if status == "sucesso":
                print(f"  ✓ Concluído: {value} ({elapsed_time:.1f}s)")

            try:
                evento, level = determine_evt_level(error_msg, timeout_msg)
                if timeout_msg and not error_msg:
                    error_msg = timeout_msg
                log_event(level, evt=evento, notebook=value, elapsed_time=elapsed_time, error=error_msg)
            except NameError:
                pass

            with results_lock:
                resultados.append({
                    "notebook": value,
                    "status": status,
                    "elapsed": elapsed_time,
                    "error": error_msg
                })
        q.task_done()


for i in range(worker_count):
    t = Thread(target=run_tasks, args=(run_notebook, q))
    t.daemon = True
    t.start()

q.join()

execution_end = datetime.now()

# --- Resumo final (len() evita conflito com pyspark.sum) ---
total = len(resultados)
ok = len([r for r in resultados if r["status"] == "sucesso"])
erros = len([r for r in resultados if r["status"] == "erro"])
touts = len([r for r in resultados if r["status"] == "timeout"])

print(f"\n{'='*55}") 
print(f"  📊 RESUMO: {ok}/{total} sucesso | {erros} erros | {touts} timeouts")
print(f"  📅 Fim: {(execution_end + _fuso_print).strftime('%d/%m/%Y %H:%M:%S')}")
print(f"{'='*55}")

# Enviar card de resumo para o Teams
try:
    enviar_resumo_teams(var_teams, resultados, worker_count, timeout, execution_start, execution_end)
except NameError:
    pass